# Build and train an MLP in PyTorch

A point $(x_1, x_2)$ in the square $[-1, 1]^2$ gets class 1 when its two coordinates have the
same sign and class 0 when they differ: the XOR pattern, which no straight line can split.
Here you write a training loop and watch a linear model fail on it, train a one-hidden-layer
MLP and plot its decision regions, then set a four-unit network's weights by hand and count
how many training runs reach a solution that good at widths 1 to 16.

**How to work through it.** Each part asks for one or two functions. Replace the `TODO` and the
`raise NotImplementedError` with your code, then run the check cell under it. The check
prints `Part N passed` when your code is right, and stops with a message that says
what is wrong when it is not.

## Setup

Run this cell first. It loads the libraries and the data every part uses.

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn as nn


def make_xor(n, seed):
    """n points uniform in [-1, 1]^2, labeled 1 when the two coordinates share a sign."""
    g = torch.Generator().manual_seed(seed)
    X = torch.rand(n, 2, generator=g) * 2 - 1
    y = (X[:, 0] * X[:, 1] > 0).long()
    return X, y


X_train, y_train = make_xor(400, seed=0)
X_test, y_test = make_xor(200, seed=1)


def accuracy(model, X, y):
    with torch.no_grad():
        return (model(X).argmax(dim=1) == y).float().mean().item()


plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=10)
plt.gca().set_aspect("equal")
plt.title("class 1 (red) where x1 and x2 share a sign")
plt.show()

## Part 1 (coding task): A training loop, and why a linear model fails

Write `train_full_batch(model, X, y, epochs=200, lr=0.05)`: train `model` with `nn.CrossEntropyLoss` and `torch.optim.Adam` at learning rate `lr`, one step per epoch on the whole of `(X, y)`, and return the list of the loss values, one per epoch.

Use it to train a linear classifier, `nn.Linear(2, 2)`, which turns each point into two class scores with no hidden layer. Call `torch.manual_seed(0)` right before building it, and report its training loss and test accuracy.

In [ ]:
def train_full_batch(model, X, y, epochs=200, lr=0.05):
    # TODO
    raise NotImplementedError


torch.manual_seed(0)
linear = nn.Linear(2, 2)
linear_losses = train_full_batch(linear, X_train, y_train)
# TODO: print the final training loss and the test accuracy

In [ ]:
assert len(linear_losses) == 200, "one loss value per epoch"
assert linear_losses[-1] < linear_losses[0], "the loss should still go down a little"
assert linear_losses[-1] > 0.65, "a linear model cannot get the loss much below log 2 = 0.693 on this data"
assert accuracy(linear, X_test, y_test) < 0.7, "a linear model should be close to guessing"
print("Part 1 passed")

## Part 2 (coding task): An MLP and its decision boundary

Write an `MLP(nn.Module)` with `__init__(self, d_hidden=16)` that holds an `nn.Sequential` named `self.net` of `nn.Linear(2, d_hidden)`, `nn.ReLU()` and `nn.Linear(d_hidden, 2)`, and a `forward` that applies it.

Then write `predict_grid(model, n=101)`: evaluate the model on an `n` by `n` grid of points covering $[-1, 1]^2$ (use `torch.linspace` and `torch.meshgrid` with `indexing="xy"`, then flatten the grid into an `(n * n, 2)` batch) and return the predicted classes as an `(n, n)` tensor, with row `i` at the `i`-th value of $x_2$ and column `j` at the `j`-th value of $x_1$.

Train `MLP(16)` with `train_full_batch` (seed 0 right before building it), report its test accuracy, and draw its decision regions with `plt.contourf` over the grid, with the training points on top.

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_hidden=16):
        super().__init__()
        # TODO
        raise NotImplementedError

    def forward(self, x):
        # TODO
        raise NotImplementedError


def predict_grid(model, n=101):
    # TODO
    raise NotImplementedError


torch.manual_seed(0)
mlp = MLP(16)
# TODO: train it, print its test accuracy, and plot the decision regions

In [ ]:
grid = predict_grid(mlp)
assert grid.shape == (101, 101), "an (n, n) grid of predicted classes"
ticks = torch.linspace(-1, 1, 101)
truth = (ticks[None, :] * ticks[:, None] > 0).long()    # row = x2, column = x1
agreement = (grid == truth).float().mean().item()
print(f"the decision regions agree with the true rule on {agreement:.1%} of the grid")
assert sum(p.numel() for p in MLP(16).parameters()) == 82, "2*16+16 + 16*2+2 = 82 parameters"
assert accuracy(mlp, X_test, y_test) > 0.9, "one hidden layer should clear 0.9"
assert agreement > 0.9, "the decision regions should follow the four quadrants"
print("Part 2 passed")

## Part 3 (coding task): Four hidden units are enough, if you set them by hand

How many hidden units does this problem need? Answer it twice: once by construction, once by training.

**By construction.** Write `hand_built_mlp()`, which returns an `MLP(4)` whose weights you set yourself inside `torch.no_grad()` (with `.copy_()` and `.zero_()`), with no training, so that it classifies every point off the axes correctly. Use the identity

$$
|x_1 + x_2| - |x_1 - x_2| \quad\begin{cases} > 0 & \text{when } x_1x_2 > 0,\\ < 0 & \text{when } x_1x_2 < 0,\end{cases}
\qquad |u| = \mathrm{ReLU}(u) + \mathrm{ReLU}(-u).
$$

**By training.** Write `width_sweep(widths, seeds=range(5))`: for every width and seed, call `torch.manual_seed(seed)`, train a fresh `MLP(width)` with `train_full_batch`, and record its test accuracy. Return a dictionary mapping each width to the list of its accuracies, and print the mean for each width in `(1, 2, 4, 8, 16)`.

In [ ]:
def hand_built_mlp():
    model = MLP(4)
    with torch.no_grad():
        # TODO: hidden unit k computes ReLU(w_k . x), and the two output scores combine them.
        raise NotImplementedError
    return model


def width_sweep(widths, seeds=range(5)):
    # TODO
    raise NotImplementedError


sweep = None    # TODO: width_sweep((1, 2, 4, 8, 16)), then print the mean accuracy for each width

In [ ]:
hand = hand_built_mlp()
X_big, y_big = make_xor(10000, seed=5)
assert accuracy(hand, X_big, y_big) == 1.0, "the hand-built network should be right on every point"
assert len(list(hand.parameters())) == 4 and hand.net[0].weight.shape == (4, 2), "use MLP(4)"
assert set(sweep) == {1, 2, 4, 8, 16} and all(len(a) == 5 for a in sweep.values()), "five runs per width"
mean = {w: sum(a) / len(a) for w, a in sweep.items()}
assert mean[1] < 0.75, "one hidden unit is a single hinge, so it cannot do much better than a line"
assert mean[16] > 0.97, "with 16 units training should succeed every time"
assert min(sweep[4]) < 0.95 and mean[4] < mean[16], "at width 4 training should sometimes miss the solution that exists"
print("Part 3 passed")

## Solution

### Part 1: A training loop, and why a linear model fails

In [ ]:
def train_full_batch(model, X, y, epochs=200, lr=0.05):
    loss_fn = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    losses = []
    for _ in range(epochs):
        loss = loss_fn(model(X), y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses.append(loss.item())
    return losses


torch.manual_seed(0)
linear = nn.Linear(2, 2)
linear_losses = train_full_batch(linear, X_train, y_train)
print(f"linear model: final training loss {linear_losses[-1]:.4f}, test accuracy {accuracy(linear, X_test, y_test):.3f}")

**Why it works.** A linear classifier predicts class 1 on one side of a single straight line. The two class-1 quadrants (++ and −−) sit diagonally opposite each other, with the class-0 quadrants between them, so any line that puts both class-1 quadrants on one side must also take in large parts of the class-0 quadrants. This is XOR on the signs, the classic function no linear boundary can separate. The best the model can do is predict roughly the majority class everywhere, so its loss stays near $\log 2 \approx 0.693$, the loss of a 50/50 guess, and its test accuracy near 0.55, the share of the larger class in the test set.

### Part 2: An MLP and its decision boundary

In [ ]:
class MLP(nn.Module):
    def __init__(self, d_hidden=16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, d_hidden),
            nn.ReLU(),
            nn.Linear(d_hidden, 2),
        )

    def forward(self, x):
        return self.net(x)


def predict_grid(model, n=101):
    ticks = torch.linspace(-1, 1, n)
    x1, x2 = torch.meshgrid(ticks, ticks, indexing="xy")      # x1 varies along columns
    points = torch.stack([x1.reshape(-1), x2.reshape(-1)], dim=1)
    with torch.no_grad():
        return model(points).argmax(dim=1).reshape(n, n)


torch.manual_seed(0)
mlp = MLP(16)
mlp_losses = train_full_batch(mlp, X_train, y_train)
print(f"MLP(16): final training loss {mlp_losses[-1]:.4f}, test accuracy {accuracy(mlp, X_test, y_test):.3f}")

ticks = torch.linspace(-1, 1, 101)
plt.contourf(ticks, ticks, predict_grid(mlp), levels=[-0.5, 0.5, 1.5], cmap="coolwarm", alpha=0.3)
plt.scatter(X_train[:, 0], X_train[:, 1], c=y_train, cmap="coolwarm", s=10)
plt.gca().set_aspect("equal")
plt.title("MLP(16) decision regions")
plt.show()

**Why it works.** Each hidden ReLU unit is a "hinge": zero on one side of a line and rising on the other. The output scores are sums of these hinges, so the decision boundary is made of straight pieces that can bend at the hinge lines. With 16 hinges the network bends its boundary along both axes and carves out the two diagonal quadrants. The only errors left are points very close to the axes, where the true boundary is sharp and the training points are sparse.

### Part 3: Four hidden units are enough, if you set them by hand

In [ ]:
def hand_built_mlp():
    model = MLP(4)
    with torch.no_grad():
        first, second = model.net[0], model.net[2]
        # Hidden units: ReLU(x1 + x2), ReLU(-x1 - x2), ReLU(x1 - x2), ReLU(-x1 + x2).
        first.weight.copy_(torch.tensor([[1.0, 1.0], [-1.0, -1.0], [1.0, -1.0], [-1.0, 1.0]]))
        first.bias.zero_()
        # Score of class 1 = |x1 + x2| - |x1 - x2|, score of class 0 = its negative.
        second.weight.copy_(torch.tensor([[-1.0, -1.0, 1.0, 1.0], [1.0, 1.0, -1.0, -1.0]]))
        second.bias.zero_()
    return model


def width_sweep(widths, seeds=range(5)):
    results = {}
    for width in widths:
        results[width] = []
        for seed in seeds:
            torch.manual_seed(seed)
            model = MLP(width)
            train_full_batch(model, X_train, y_train)
            results[width].append(accuracy(model, X_test, y_test))
    return results


sweep = width_sweep((1, 2, 4, 8, 16))
for width, accs in sweep.items():
    print(f"width {width:2d}: mean test accuracy {sum(accs) / len(accs):.3f}   runs {[round(a, 3) for a in accs]}")

**Why it works.** $(x_1 + x_2)^2 - (x_1 - x_2)^2 = 4x_1x_2$, so $|x_1 + x_2| - |x_1 - x_2|$ has the sign of $x_1x_2$, and each absolute value takes two ReLU units. Four hidden units are therefore enough to classify every point perfectly. Yet training finds a perfect solution with four units only in some runs (mean accuracy about 0.89): from a random start, gradient descent can settle where some hinges point the wrong way or switch off for good. With 8 or 16 units almost every run succeeds. A network wider than the minimum is not wasted: the extra units give gradient descent many more ways to reach a good solution, which is one reason practical networks are much larger than the smallest network that could represent the answer.